In [124]:
import numpy as np
from kaggle_environments import make,evaluate
import random
import math
import base64
import inspect

import torch
import torch.nn as nn
import torch.optim as optim

In [106]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

In [107]:
env = make('connectx', debug=False)
config = env.configuration
config

{'episodeSteps': 1000,
 'actTimeout': 2,
 'runTimeout': 1200,
 'columns': 7,
 'rows': 6,
 'inarow': 4,
 'agentTimeout': 60,
 'timeout': 2}

In [108]:
env.reset()

[{'action': 0,
  'reward': 0,
  'info': {},
  'observation': {'remainingOverageTime': 60,
   'step': 0,
   'board': [0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0],
   'mark': 1},
  'status': 'ACTIVE'},
 {'action': 0,
  'reward': 0,
  'info': {},
  'observation': {'remainingOverageTime': 60, 'mark': 2},
  'status': 'INACTIVE'}]

In [109]:
class DQNA:
    def __init__(self, rows, columns, action_size, inarow):
        self.rows = rows
        self.columns = columns
        self.action_size = action_size
        self.memory = []
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_max = 0.9
        self.epsilon_decay = 2500
        self.learning_rate = 0.001
        self.tau = 0.005

        self.policy_net = nn.Sequential(
            nn.Conv2d(in_channels = 1, out_channels = 64, kernel_size = inarow, padding = 'same'),
            nn.ReLU(),
            nn.Dropout(),
            nn.Conv2d(in_channels = 64, out_channels = 64, kernel_size = inarow),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(in_features=64*(rows - inarow + 1)*(columns - inarow + 1), out_features=64),
            nn.ReLU(),
            nn.Linear(in_features=64, out_features=action_size),
            nn.ReLU()
        )
        self.target_net = nn.Sequential(
            nn.Conv2d(in_channels = 1, out_channels = 64, kernel_size = inarow, padding = 'same'),
            nn.ReLU(),
            nn.Dropout(),
            nn.Conv2d(in_channels = 64, out_channels = 64, kernel_size = inarow),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(in_features=64*(rows - inarow + 1)*(columns - inarow + 1), out_features=64),
            nn.ReLU(),
            nn.Linear(in_features=64, out_features=action_size),
            nn.ReLU()
        )
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.criterion = nn.MSELoss()
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=self.learning_rate, amsgrad=True)

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))
        if len(self.memory) > 10000:
            self.memory.pop(0)
    def act(self, state, valid_actions):
        if np.random.rand() <= self.epsilon:
            return random.choice(valid_actions)
        state = torch.from_numpy(np.expand_dims(state, axis=(0,1)))
        with torch.no_grad():
            q_values = self.policy_net(state)[0]
        q_valid = [(a,q_values[a]) for a in valid_actions]
        return max(q_valid, key=lambda x:x[1])[0]
    def replay(self, batch_size=64):
        if len(self.memory) < batch_size:
            return
        minibatch = random.sample(self.memory, batch_size)
        states = np.array([m[0] for m in minibatch], dtype=np.float32)
        actions = np.array([m[1] for m in minibatch], dtype=np.float32)
        rewards = np.array([[m[2]]*self.action_size for m in minibatch], dtype=np.float32)
        next_states = np.array([m[3] for m in minibatch], dtype=np.float32)
        dones = np.array([m[4] for m in minibatch])
        
        non_final_mask = torch.tensor(tuple(map(lambda done: done is not True,
                                        dones)), device=device, dtype=torch.bool)
        non_final_next_states = torch.cat([torch.from_numpy(np.expand_dims(s,axis=(0,1))) for s,done in zip(next_states,dones)
                                        if done is not True])     
        state_action_values = self.policy_net(torch.from_numpy(np.expand_dims(states, axis=(1)))).to(device)
        next_state_values = torch.zeros_like(state_action_values, device=device)
        with torch.no_grad():
            next_state_values[non_final_mask] = self.target_net(non_final_next_states).to(device)
        expected_state_action_values = (next_state_values * self.gamma) + torch.from_numpy(rewards).to(device)
        loss = self.criterion(state_action_values.unsqueeze(1), expected_state_action_values.unsqueeze(1))
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_value_(self.policy_net.parameters(), 100)
        self.optimizer.step()
    def update_epsilon(self, steps_done):
        self.epsilon = self.epsilon_min + (self.epsilon_max - self.epsilon_min) * \
        math.exp(-1. * steps_done / self.epsilon_decay)
        return steps_done+1
    def clone_to_target(self):
        target_net_state_dict = self.target_net.state_dict()
        policy_net_state_dict = self.policy_net.state_dict()
        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key]*self.tau + target_net_state_dict[key]*(1-self.tau)
        self.target_net.load_state_dict(target_net_state_dict)

In [ ]:
def get_valid_moves(board, config):
    return [c for c in range(config.columns) if board[c]==0]
def state_to_np(board,config):
    return np.array(board).reshape(config.rows, config.columns).astype(np.float32)
def check_window(window,num_discs, piece, config):
    return (window.count(piece) == num_discs and window.count(0) == config.inarow - num_discs)
def count_windows(grid, num_discs, piece, config):
    num_windows = 0
    grid_2d = np.array(grid).reshape(config.rows, config.columns)
    for r in range(config.rows):
        for c in range(config.columns - config.inarow + 1):
            window = list(grid_2d[r,c:c+config.inarow])
            if check_window(window,num_discs,piece,config): num_windows += 1
    for r in range(config.rows - config.inarow + 1):
        for c in range(config.columns):
            window = list(grid_2d[r:r+config.inarow,c])
            if check_window(window,num_discs,piece,config): num_windows += 1
    for r in range(config.rows - config.inarow + 1):
        for c in range(config.columns - config.inarow + 1):
            window = list(grid_2d[r:r+config.inarow,c:c+config.inarow].diagonal())
            if check_window(window,num_discs,piece,config): num_windows += 1
            window = list(np.flipud(grid_2d[r:r+config.inarow,c:c+config.inarow]).diagonal())
            if check_window(window,num_discs,piece,config): num_windows += 1
    return num_windows
def get_shaped_reward(board,action,step_reward,config,done,mark):
    if done:
        if step_reward == 1:
            return 1.0
        elif step_reward == -1:
            return -1.0
        else:
            return 0.0
    opponent_mark = 1 if mark == 2 else 2
    my_threes = count_windows(board,3,mark,config)
    opp_threes = count_windows(board,3,opponent_mark,config)
    heuristic_score = (my_threes * 0.1) - (opp_threes * 0.2)
    if action == config.columns // 2:
        heuristic_score+=0.05
    return max(-0.8,min(0.8, heuristic_score))

In [ ]:
if torch.cuda.is_available() or torch.backends.mps.is_available():
    num_episodes = 200000
else:
    num_episodes = 2000
dqn_agent = DQNA(
    rows=config.rows,
    columns=config.columns,
    action_size=config.columns,
    inarow=config.inarow
)
print(f'Starting training...\nEpisode 0/{num_episodes}')
trainer = env.train([None, "random"])
total_reward = 0
steps_done = 0
for ep in range(num_episodes):
    obs = trainer.reset()
    board = obs['board']
    state = state_to_np(board,config)
    done = False
    while not done:
        valid_moves = get_valid_moves(board,config)
        action = dqn_agent.act(state,valid_moves)
        steps_done = dqn_agent.update_epsilon(steps_done)
        next_obs,step_reward,done,info = trainer.step(int(action))
        next_board = next_obs['board']
        mark = next_obs['mark']
        current_reward = get_shaped_reward(next_board, action, step_reward, config, done, mark)
        next_state = state_to_np(next_board,config)
        dqn_agent.remember(state,action,current_reward,next_state,done)
        state = next_state
        board = next_board
        total_reward += current_reward
        dqn_agent.replay()
        dqn_agent.clone_to_target()
    if (ep+1) % 50 == 0:
        print(f"Episode {ep+1}/{num_episodes}, Total Reward = {total_reward:.2f}, Epsilon = {dqn_agent.epsilon:.2f}")
        total_reward = 0


Starting training...
Episode 0/2000
Episode 50/2000, Total Reward = -5.15, Epsilon = 0.74
Episode 100/2000, Total Reward = -28.05, Epsilon = 0.59
Episode 150/2000, Total Reward = -50.70, Epsilon = 0.48
Episode 200/2000, Total Reward = -33.40, Epsilon = 0.40
Episode 250/2000, Total Reward = -34.85, Epsilon = 0.32
Episode 300/2000, Total Reward = -17.75, Epsilon = 0.27
Episode 350/2000, Total Reward = -7.80, Epsilon = 0.23
Episode 400/2000, Total Reward = -20.55, Epsilon = 0.19
Episode 450/2000, Total Reward = 8.95, Epsilon = 0.17
Episode 500/2000, Total Reward = -10.85, Epsilon = 0.14
Episode 550/2000, Total Reward = -15.10, Epsilon = 0.12
Episode 600/2000, Total Reward = 7.75, Epsilon = 0.10
Episode 650/2000, Total Reward = -15.10, Epsilon = 0.08
Episode 700/2000, Total Reward = -3.25, Epsilon = 0.07
Episode 750/2000, Total Reward = 17.15, Epsilon = 0.06
Episode 800/2000, Total Reward = -0.05, Epsilon = 0.06
Episode 850/2000, Total Reward = 29.80, Epsilon = 0.05
Episode 900/2000, Total

In [121]:
PATH = 'connectX.pt'
torch.save(dqn_agent.policy_net.state_dict(), 'connectX.pt')
print("Model saved")

Model saved


In [136]:
OUTPUT_PATH = 'submission.py'
no_params_path = 'submission_without_parameters.py'


def agent_function(observation, configuration):

    import os

    import io
    import numpy as np
    import random
    import base64
    import torch
    import torch.nn as nn

    def build_model(inarow, rows, columns, action_size):
        return nn.Sequential(
            nn.Conv2d(in_channels = 1, out_channels = 64, kernel_size = inarow, padding = 'same'),
            nn.ReLU(),
            nn.Dropout(),
            nn.Conv2d(in_channels = 64, out_channels = 64, kernel_size = inarow),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(in_features=64*(rows - inarow + 1)*(columns - inarow + 1), out_features=64),
            nn.ReLU(),
            nn.Linear(in_features=64, out_features=action_size),
            nn.ReLU()
        )

    model = build_model(
        rows=configuration.rows,
        columns=configuration.columns,
        action_size=configuration.columns,
        inarow=configuration.inarow
    )
    encoded_weights = """
    BASE64_PARAMS"""

    decoded = base64.b64decode(encoded_weights)
    buffer = io.BytesIO(decoded)
    model.load_state_dict(torch.load(buffer))


    def check_win(board, mark, config):
        columns = config.columns
        rows = config.rows
        
        def count(offset_row, offset_col):
            for r in range(rows):
                for c in range(columns):
                    if r + 3 * offset_row >= rows or r + 3 * offset_row < 0: continue
                    if c + 3 * offset_col >= columns or c + 3 * offset_col < 0: continue
                    if board[r * columns + c] == mark and board[(r + offset_row) * columns + (c + offset_col)] == mark and board[(r + 2 * offset_row) * columns + (c + 2 * offset_col)] == mark and board[(r + 3 * offset_row) * columns + (c + 3 * offset_col)] == mark:
                        return True
            return False
        
        return count(1, 0) or count(0, 1) or count(1, 1) or count(1, -1)

    def simulate_drop(board, col, mark, config):
        next_board = list(board)
        for row in range(config.rows-1, -1, -1):
            if next_board[row * config.columns + col] == 0:
                next_board[row * config.columns + col] = mark
                break
        return next_board

    board = observation.board
    mark = observation.mark
    opponent = 1 if mark == 2 else 2
        
    valid_moves = [c for c in range(configuration.columns) if board[c] == 0]
    if len(valid_moves) == 0: return 0
        
    for col in valid_moves:
        next_board = simulate_drop(board, col, mark, configuration)
        if check_win(next_board, mark, configuration):
            return col
                
    for col in valid_moves:
        next_board = simulate_drop(board, col, opponent, configuration)
        if check_win(next_board, opponent, configuration):
            return col
    
    state = np.array(board).reshape(configuration.rows, configuration.columns).astype(np.float32)
    with torch.no_grad():
        q_values = model(torch.from_numpy(np.expand_dims(state, axis=(0,1))))[0]
    
    q_valid = [(c, q_values[c]) for c in valid_moves]
    best_action = max(q_valid, key=lambda x: x[1])[0]
    
    return int(best_action)

def write_agent_to_file(function, file):
    with open(file, "w") as f:
        f.write(inspect.getsource(function))
        print(function, "written to", file)

write_agent_to_file(agent_function, no_params_path)

with open(PATH, 'rb') as f:
    raw_bytes = f.read()
    encoded_weights = base64.encodebytes(raw_bytes).decode()

with open(no_params_path, 'r') as file:
    data = file.read()

data = data.replace('BASE64_PARAMS', encoded_weights)

with open(OUTPUT_PATH, 'w') as f:
    f.write(data)
    print('written agent with net parameters to', OUTPUT_PATH)

print(f"{OUTPUT_PATH} successfully created!")

<function agent_function at 0x7f6d1f71a200> written to submission_without_parameters.py
written agent with net parameters to submission.py
submission.py successfully created!


In [137]:
import sys
from kaggle_environments import evaluate, make, utils, agent
out = sys.stdout
submission = utils.read_file(OUTPUT_PATH)
saved_agent = agent.get_last_callable(submission)
sys.stdout = out

env = make("connectx", debug=True)
env.run([saved_agent, saved_agent])
print("Success!" if env.state[0].status == env.state[1].status == "DONE" else "Failed...")

Success!
